# 05. Generate DB2 Dataset (2000 Hz)
**Objective:** Unzip and compile NinaPro DB2 into 250ms windows (500 samples) with 50ms strides (100 samples), replicating the exact preprocessing pipeline from Molina et al. (2025).

In [6]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import zipfile
import numpy as np
import scipy.io as sio

sys.path.append(os.path.abspath('../'))
from src.config import RAW_DATA_DIR, PREPROCESSED_DIR
from src.preprocess import sEMGPreprocessor

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1. Unzip and Load DB2 Data

In [8]:
zip_path = os.path.join(RAW_DATA_DIR, "Ninapro_DB2", "DB2_s1.zip")
extract_dir = os.path.join(RAW_DATA_DIR, "Ninapro_DB2", "s1")

if not os.path.exists(extract_dir):
    print(f"Extracting {zip_path}...")
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)

exercises = ['E1', 'E2', 'E3']

emg_list, labels_list, reps_list = [], [], []

print("Loading raw DB2 .mat files...")
for ex in exercises:
    mat_path = os.path.join(extract_dir, f"DB2_s1/S1_{ex}_A1.mat")
    mat_data = sio.loadmat(mat_path)
    
    # DB2 labels are natively continuous (1-49), so NO OFFSETS ARE NEEDED.
    labels = mat_data['restimulus'].squeeze().copy()
    
    emg_list.append(mat_data['emg'])
    labels_list.append(labels)
    reps_list.append(mat_data['rerepetition'].squeeze())

emg_full = np.vstack(emg_list)
labels_full = np.concatenate(labels_list)
reps_full = np.concatenate(reps_list)

print(f"Raw Continuous DB2 sEMG Shape: {emg_full.shape}")
print(f"Total Labels: {len(np.unique(labels_full))} (Expected: 50 including Rest)")

Loading raw DB2 .mat files...
Raw Continuous DB2 sEMG Shape: (5238693, 12)
Total Labels: 50 (Expected: 50 including Rest)


### 2. Filter, Standardize, and Window (Molina Specifications)

In [9]:
# DB2 is sampled at 2000 Hz
preprocessor = sEMGPreprocessor(sample_rate=2000)

print("\nApplying 20-500Hz Band-Pass Filter...")
filtered_emg = preprocessor.filter_signal(emg_full, btype='bandpass', lowcut=20.0, highcut=500.0, order=4)

print("Standardizing channels...")
norm_emg = preprocessor.fit_standardize(filtered_emg)

# Molina's exact windowing: 250ms windows, 50ms strides
win_size = 500
stride = 100
X_dense, y_dense, reps_dense = preprocessor.extract_dense_windows(norm_emg, labels_full, reps_full, win_size, stride)

print("\nBalancing Rest Class...")
X_bal, y_bal, reps_bal = preprocessor.balance_rest_class(X_dense, y_dense, reps_dense)


Applying 20-500Hz Band-Pass Filter...
Standardizing channels...
Extracting windows (size=500 samples, stride=100 samples)...

Balancing Rest Class...


### 3. Serialize to HDF5

In [10]:
output_path = os.path.join(PREPROCESSED_DIR, "DB2_S1_Windows.h5")

with h5py.File(output_path, 'w') as f:
    f.create_dataset('X', data=X_bal, compression='gzip')
    f.create_dataset('y', data=y_bal)
    f.create_dataset('reps', data=reps_bal)

print(f"\nSuccess! DB2 Dataset saved to: {output_path}")
print(f"Final Tensor Shape: {X_bal.shape}")


Success! DB2 Dataset saved to: /workspaces/TCC/data/preprocessed/DB2_S1_Windows.h5
Final Tensor Shape: (24463, 500, 12)
